In [1]:
#importing the libraries
import os 
import fastf1
import pandas as pd

In [2]:
#cache forlder 
cache_folder = '../../fastf1_pipeline/cache_folder'
if not os.path.exists(cache_folder):
    os.makedirs(cache_folder)

fastf1.Cache.enable_cache(cache_folder)

/usr/local/lib/python3.13/site-packages/fastf1/req.py:251: FutureWarning: `.enable_cache` is deprecated and will be removed in a future version. Use `.configure` instead.
  warnings.warn("`.enable_cache` is deprecated and will be removed in a "


In [3]:
#get data from the folder 
data_folder = "../../data"  # Ergast CSVs, see data/README.md


In [4]:
# Load CSV files
circuits = pd.read_csv(f"{data_folder}/circuits.csv")
constructors = pd.read_csv(f"{data_folder}/constructors.csv")
drivers = pd.read_csv(f"{data_folder}/drivers.csv")
races = pd.read_csv(f"{data_folder}/races.csv")
results = pd.read_csv(f"{data_folder}/results.csv")
lap_times = pd.read_csv(f"{data_folder}/lap_times.csv")
pit_stops = pd.read_csv(f"{data_folder}/pit_stops.csv")
qualifying = pd.read_csv(f"{data_folder}/qualifying.csv")
seasons = pd.read_csv(f"{data_folder}/seasons.csv")
status = pd.read_csv(f"{data_folder}/status.csv")


In [5]:
#sanity check
print("Races:\n", races.head(), "\n")
print("Drivers:\n", drivers.head(), "\n")
print("Results:\n", results.head(), "\n")


Races:
    raceId  year  round  circuitId                   name        date  \
0       1  2009      1          1  Australian Grand Prix  2009-03-29   
1       2  2009      2          2   Malaysian Grand Prix  2009-04-05   
2       3  2009      3         17     Chinese Grand Prix  2009-04-19   
3       4  2009      4          3     Bahrain Grand Prix  2009-04-26   
4       5  2009      5          4     Spanish Grand Prix  2009-05-10   

       time                                                url fp1_date  \
0  06:00:00  http://en.wikipedia.org/wiki/2009_Australian_G...       \N   
1  09:00:00  http://en.wikipedia.org/wiki/2009_Malaysian_Gr...       \N   
2  07:00:00  http://en.wikipedia.org/wiki/2009_Chinese_Gran...       \N   
3  12:00:00  http://en.wikipedia.org/wiki/2009_Bahrain_Gran...       \N   
4  12:00:00  http://en.wikipedia.org/wiki/2009_Spanish_Gran...       \N   

  fp1_time fp2_date fp2_time fp3_date fp3_time quali_date quali_time  \
0       \N       \N       \N       \

In [6]:
#to find the races in the united states only
us_gp = races[races["name"].str.contains("united states", case=False, na=False)]
print (us_gp[["raceId", "year", "round", "name", "date"]].to_string(index=False))

 raceId  year  round                          name       date
     42  2007      7      United States Grand Prix 2007-06-17
     62  2006     10      United States Grand Prix 2006-07-02
     79  2005      9      United States Grand Prix 2005-06-19
     98  2004      9      United States Grand Prix 2004-06-20
    122  2003     15      United States Grand Prix 2003-09-28
    139  2002     16      United States Grand Prix 2002-09-29
    156  2001     16      United States Grand Prix 2001-09-30
    172  2000     15      United States Grand Prix 2000-09-24
    305  1991      1      United States Grand Prix 1991-03-10
    321  1990      1      United States Grand Prix 1990-03-11
    360  1989      5      United States Grand Prix 1989-06-04
    453  1983      2 United States Grand Prix West 1983-03-27
    469  1982      3 United States Grand Prix West 1982-04-04
    483  1981      1 United States Grand Prix West 1981-03-15
    501  1980      4 United States Grand Prix West 1980-03-30
    511 

In [7]:
austin_gp = races[races["name"].str.contains("United States Grand Prix", case=False, na=False)]

print(austin_gp[["raceId", "year", "round", "name", "date"]].to_string(index=False))

 raceId  year  round                          name       date
     42  2007      7      United States Grand Prix 2007-06-17
     62  2006     10      United States Grand Prix 2006-07-02
     79  2005      9      United States Grand Prix 2005-06-19
     98  2004      9      United States Grand Prix 2004-06-20
    122  2003     15      United States Grand Prix 2003-09-28
    139  2002     16      United States Grand Prix 2002-09-29
    156  2001     16      United States Grand Prix 2001-09-30
    172  2000     15      United States Grand Prix 2000-09-24
    305  1991      1      United States Grand Prix 1991-03-10
    321  1990      1      United States Grand Prix 1990-03-11
    360  1989      5      United States Grand Prix 1989-06-04
    453  1983      2 United States Grand Prix West 1983-03-27
    469  1982      3 United States Grand Prix West 1982-04-04
    483  1981      1 United States Grand Prix West 1981-03-15
    501  1980      4 United States Grand Prix West 1980-03-30
    511 

In [8]:
austin_gp = races[
    (races["name"].str.contains("United States Grand Prix", case=False, na=False))
    & ~races["name"].str.contains("West", case=False, na=False)
    & (races["year"] >= 2012)
]

print(austin_gp[["raceId", "year", "round", "name", "date"]].to_string(index=False))


 raceId  year  round                     name       date
    878  2012     19 United States Grand Prix 2012-11-18
    898  2013     18 United States Grand Prix 2013-11-17
    916  2014     17 United States Grand Prix 2014-11-02
    942  2015     16 United States Grand Prix 2015-10-25
    965  2016     18 United States Grand Prix 2016-10-23
    985  2017     17 United States Grand Prix 2017-10-22
   1006  2018     18 United States Grand Prix 2018-10-21
   1028  2019     19 United States Grand Prix 2019-11-03
   1069  2021     17 United States Grand Prix 2021-10-24
   1093  2022     19 United States Grand Prix 2022-10-23
   1116  2023     18 United States Grand Prix 2023-10-22
   1139  2024     19 United States Grand Prix 2024-10-20


In [9]:
#load the result data

results = pd.read_csv(f"{data_folder}/results.csv")

#merge austin_gp and the results.csv
austin_results = pd.merge (
    austin_gp, results, on="raceId", how= "inner"
)

# Keep winners only
austin_winners = austin_results[austin_results["position"] == "1"]

# load drivers data to load drivers name
drivers = pd.read_csv(f"{data_folder}/drivers.csv")

# merge winners with drivers info

austin_winners = pd.merge(
    austin_winners, drivers, on="driverId", how="inner"
)

#print the driver, year, race and the winner

print(
    austin_winners[["year","name", "date", "forename", "surname"]]
    .sort_values("year")
    .to_string(index=False)
)

 year                     name       date  forename    surname
 2012 United States Grand Prix 2012-11-18     Lewis   Hamilton
 2013 United States Grand Prix 2013-11-17 Sebastian     Vettel
 2014 United States Grand Prix 2014-11-02     Lewis   Hamilton
 2015 United States Grand Prix 2015-10-25     Lewis   Hamilton
 2016 United States Grand Prix 2016-10-23     Lewis   Hamilton
 2017 United States Grand Prix 2017-10-22     Lewis   Hamilton
 2018 United States Grand Prix 2018-10-21      Kimi  Räikkönen
 2019 United States Grand Prix 2019-11-03  Valtteri     Bottas
 2021 United States Grand Prix 2021-10-24       Max Verstappen
 2022 United States Grand Prix 2022-10-23       Max Verstappen
 2023 United States Grand Prix 2023-10-22       Max Verstappen
 2024 United States Grand Prix 2024-10-20   Charles    Leclerc
